# 38 · 系统工程：Serving / 可观测性 / 监控

> Demo 能跑不算数，**上线后还能被看懂、被追踪、被回滚**才算工程。这是 RAG 走向生产的临门一脚。

**本文件覆盖知识点**：System Engineering / Serving / FastAPI 部署 / 观测 / Tracing / Logging / Metrics / CI-CD / 版本管理与回滚

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. Serving：把 RAG 包成服务

最常用的形态：**FastAPI 提供 HTTP 接口**，前后端/其它服务调用。

```text
RAG 服务(进程常驻: 已加载向量索引 + 连接百炼)
   POST /chat        ← 问答
   POST /index/upsert← 新增知识(重新切分+embedding+入库)
```

### Serving 三件套
- **进程常驻**：索引/模型只加载一次，别每次请求重建；
- **异步/并发**：检索与 LLM 调用用异步，提升吞吐；
- **优雅降级**：LLM 超时降级到“仅返回检索片段”，索引挂了降级到“仅生成”。

In [ ]:
# 用 uuid 演示给每个请求分配唯一 trace_id —— 追踪的地基
import uuid

def start_request(user, question):
    trace_id = uuid.uuid4().hex[:12]
    print(f'[LOG] trace={trace_id} user={user} q={question[:20]}...')  # Logging
    return trace_id

def span(name, trace_id, ms):
    print(f'[SPAN] trace={trace_id} step={name} cost={ms}ms')          # Tracing

tid = start_request('u_1001', '星云私有化怎么部署')
span('embed',  tid, 32)
span('search', tid, 18)
span('rerank', tid, 60)
span('generate', tid, 1200)
print('\n=> 每个请求一个 trace_id，整条链路(检索/重排/生成)耗时一目了然。')

## 2. Observability（可观测性）= Logging + Tracing + Metrics

| 支柱 | 回答什么 | 常见载体 |
|------|---------|---------|
| **Logging 日志** | 发生了什么 | 结构化日志 |
| **Tracing 链路** | 一个请求内部经历了什么 | OpenTelemetry / LangSmith |
| **Metrics 指标** | 系统整体健康度 | Prometheus / Grafana |

### RAG 关键指标
- 端到端延迟 / TTFT / P95；
- 检索命中率、空检索率（答非所问的根源）；
- 无引用占比、用户负面反馈率（答案质量代理）；
- 成本：token 消耗/请求。

> **生产必备习惯**：每次问答都落一条带 trace_id 的记录，保存“问题+检索片段+回答+耗时”，日后可回溯分析失败案例。

In [ ]:
# 知识点·真调说明：可观测性排障 —— 把一条真实报错交给模型翻译成“值班排查建议”
_llm_live(
    prompt='下面是一条 RAG 服务上线后的线上报错，请像值班工程师一样给出排查建议：\n'
           '2026-09-06 10:31:02 ERROR  POST /chat -> 500  trace=a1f2c3\n'
           '  [SPAN] trace=a1f2c3 step=retrieve cost=12ms ok\n'
           '  [SPAN] trace=a1f2c3 step=rerank  cost=9ms  ok\n'
           '  [SPAN] trace=a1f2c3 step=generate cost=28001ms timeout(>28000)\n'
           '  dashscope.api: RateLimitError 429 qwen-plus\n'
           '同期监控：P95 端到端延迟 1.2s→9.8s；空检索率 2%→15%；该报错只在最新 prompt 版本下高频出现。\n'
           '请分 3 点回答：①最可能根因 ②怎么验证 ③临时止血/降级方案。每点不超过 2 句话。',
    system='你是 RAG 服务的值班工程师(SRE)，回答务实、可直接执行，用 ①②③ 列点，不绕圈子。',
    fallback='未配置 Key 的固定样例：\n'
             '① 最可能根因：新 prompt 把检索片段/引用拼得过长，generate 输入 token 暴涨，'
             '单次超过 28s 超时并叠加 429 限流重试，把 P95 拖到 9.8s；空检索率上升指向检索/改写参数也变了。\n'
             '② 怎么验证：对比新旧 prompt 的输入 token 数与 generate 耗时，确认 429 重试次数，'
             '再查该版本对检索改写/截断的改动。\n'
             '③ 止血：先回滚 prompt 到上一版本；同时给 generate 加超时熔断，降级到“仅返回检索片段”，'
             '并给 429 退避重试、放开并发上限。',
    temperature=0.2,
)
print('→ 模型读的不是源码，而是 Logging/Tracing/Metrics 留下的“痕迹”——'
      '能定位到 generate/429/新版本，正是可观测性三支柱 + 版本化的价值。')

## 3. 上线流程：CI/CD 与回滚

```text
代码/提示词/索引变更
   → CI(测试: 单测+评估集回归)     ← 指标掉就拦住
   → 构建镜像/版本化
   → CD(灰度放量: 1% → 50% → 100%)
   → 监控指标异常 → 回滚到上个版本
```

- 提示词与索引也做**版本管理**（评估集固定才可回归）；
- 新知识上线前先在 staging 库评估，别直接污染生产索引。



In [ ]:
# 知识点·真调说明：版本管理与回滚决策 —— 让模型对“指标回退 vs 主观好评”做灰度决策
_llm_live(
    prompt='你是 RAG 服务的发布负责人。新 prompt v2（改了客服口吻、新增引用格式）昨天灰度到 10% 流量，对照数据：\n'
           '- 固定评测集 Recall 0.82 → 0.71（下降），Faithfulness 0.90 → 0.88\n'
           '- 用户负面反馈率 3.1% → 8.6%\n'
           '- 抽样访谈 6/10 人觉得 v2“更像人、更愿意继续用”\n'
           '- v2 的新引用格式与现有溯源组件不兼容，无法在 v2 内单独回退格式\n'
           '请给出决策：继续放量 / 回滚 / 先改再上？为什么？分 3 点，每点不超过 2 句话。',
    system='你是资深发布/可靠性负责人，先给结论再讲依据，决策以数据为准而非主观感受。',
    fallback='未配置 Key 的固定样例：\n'
             '① 结论：先回滚到 v1，再把“风格优化”与“引用格式改动”拆成两个独立版本分别验证。\n'
             '② 依据：Recall/Faithfulness 双降、负面反馈翻倍是核心指标回退，“感觉更好”不能作为上线依据；'
             '风格是锦上添花，不应搭车破坏可回归的引用格式。\n'
             '③ 执行：回滚后跑一遍固定评测集确认基线复原；风格以纯提示词小版本单独灰度，'
             '绑定评测集 + 溯源格式回归通过后再逐步放量。',
    temperature=0.2,
)
print('→ 提示词/索引版本化 + 固定评测集，回滚才有“数据依据”而不是拍脑袋——CI 回归拦的就是这类回退。')

## 小结

- **Serving**：常驻进程 + 异步 + 优雅降级；
- **可观测**：Logging/Tracing/Metrics 三支柱，trace_id 贯穿全程；
- **上线**：CI 回归评估 → 灰度 → 可回滚，把 RAG 当正式软件管。